# Interactive Stock Market Analysis Tool
## Python Data Analysis Notebook

**Course:** ACC102 - Python Data Product  
**Assignment:** Mini Assignment (Track 4 - Interactive Data Analysis Tool)  
**Date:** April 2026  

### Objective
This notebook demonstrates the complete analytical workflow for comparing technology company stock performance using Python. The analysis includes data acquisition, cleaning, transformation, technical indicator calculation, and visualization.

## 1. Problem Definition and Data Relevance

### Analytical Problem
How can investors and financial analysts effectively compare the stock performance of major technology companies to make informed investment decisions?

### Target User
- Individual investors seeking to understand tech stock trends
- Financial analysts comparing company performance
- Economics and finance students learning technical analysis

### Business Context
The technology sector is a major component of global equity markets. Understanding relative stock performance, volatility, and technical indicators is crucial for:
- Portfolio construction and diversification
- Risk assessment
- Trend identification
- Comparative valuation

## 2. Data Acquisition

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

### Data Source Information

**Source:** Yahoo Finance API via `yfinance` Python library  
**Data Type:** Historical stock prices (Open, High, Low, Close, Volume)  
**Companies Analyzed:** Apple (AAPL), Microsoft (MSFT), Tesla (TSLA), NVIDIA (NVDA), Amazon (AMZN), Google (GOOGL), Meta (META), Intel (INTC)  
**Time Period:** 6 months (180 days from today)  
**Access Date:** April 15, 2026  

Yahoo Finance is selected as the data source because:
1. **Reliability**: Widely used in financial analysis and education
2. **Accessibility**: Free API with no authentication required
3. **Completeness**: Includes OHLCV data for all major stocks
4. **Timeliness**: Updated daily during market hours

In [ ]:
# Define companies and their ticker symbols
companies = {
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Tesla": "TSLA",
    "NVIDIA": "NVDA",
    "Amazon": "AMZN",
    "Google": "GOOGL",
    "Meta": "META",
    "Intel": "INTC"
}

# Define time period (6 months)
end_date = datetime.now()
start_date = end_date - timedelta(days=180)

print(f"Analysis Period: {start_date.date()} to {end_date.date()}")
print(f"Total Days: {(end_date - start_date).days}")
print(f"\nCompanies to analyze: {len(companies)}")
for company, ticker in companies.items():
    print(f"  - {company}: {ticker}")

In [ ]:
# Fetch stock data for all companies
stock_data = {}
fetch_errors = []

print("Fetching stock data...\n")

for company, ticker in companies.items():
    try:
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        stock_data[company] = data
        print(f"✓ {company} ({ticker}): {len(data)} records fetched")
    except Exception as e:
        fetch_errors.append((company, str(e)))
        print(f"✗ {company} ({ticker}): Error - {e}")

print(f"\nSuccessfully fetched data for {len(stock_data)} companies")
if fetch_errors:
    print(f"Errors encountered: {len(fetch_errors)}")

In [ ]:
# Inspect data structure
print("Sample data for Apple (first 5 rows):")
print(stock_data["Apple"].head())
print(f"\nData shape: {stock_data['Apple'].shape}")
print(f"Data types:\n{stock_data['Apple'].dtypes}")
print(f"\nMissing values:\n{stock_data['Apple'].isnull().sum()}")

## 3. Data Cleaning and Preparation

In [ ]:
# Check for missing values across all companies
print("Missing Values Summary:")
print("=" * 50)

for company, data in stock_data.items():
    missing_count = data.isnull().sum().sum()
    if missing_count > 0:
        print(f"{company}: {missing_count} missing values")
        print(data.isnull().sum())
    else:
        print(f"{company}: No missing values ✓")

In [ ]:
# Data cleaning function
def clean_stock_data(data):
    """
    Clean stock data by:
    1. Removing rows with missing values
    2. Ensuring data types are correct
    3. Verifying data consistency
    """
    # Remove rows with any missing values
    data = data.dropna()
    
    # Ensure numeric data types
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
    
    # Remove any rows created by coercion errors
    data = data.dropna()
    
    # Verify logical consistency (High >= Low, High >= Close, Low <= Close)
    assert (data['High'] >= data['Low']).all(), "High < Low detected"
    assert (data['High'] >= data['Close']).all() or (data['High'] < data['Close']).sum() == 0, "High < Close detected"
    assert (data['Low'] <= data['Close']).all() or (data['Low'] > data['Close']).sum() == 0, "Low > Close detected"
    
    return data

# Apply cleaning to all stock data
print("Cleaning stock data...\n")

for company in stock_data:
    original_length = len(stock_data[company])
    stock_data[company] = clean_stock_data(stock_data[company])
    cleaned_length = len(stock_data[company])
    removed = original_length - cleaned_length
    
    if removed > 0:
        print(f"{company}: Removed {removed} rows (original: {original_length}, cleaned: {cleaned_length})")
    else:
        print(f"{company}: No rows removed (total: {cleaned_length})")

print("\nData cleaning completed successfully")

## 4. Data Transformation and Feature Engineering

In [ ]:
# Calculate technical indicators
def add_technical_indicators(data):
    """
    Add technical indicators to stock data:
    - Moving Averages (20-day, 50-day)
    - Daily Returns
    - Price Range
    """
    data = data.copy()
    
    # Moving Averages
    data['MA20'] = data['Close'].rolling(window=20).mean()
    data['MA50'] = data['Close'].rolling(window=50).mean()
    
    # Daily Returns (percentage change)
    data['Daily_Return'] = data['Close'].pct_change() * 100
    
    # Price Range
    data['Price_Range'] = data['High'] - data['Low']
    data['Price_Range_Pct'] = (data['Price_Range'] / data['Close']) * 100
    
    # Volume Moving Average
    data['Volume_MA20'] = data['Volume'].rolling(window=20).mean()
    
    return data

# Apply technical indicators to all companies
print("Calculating technical indicators...\n")

for company in stock_data:
    stock_data[company] = add_technical_indicators(stock_data[company])
    print(f"✓ {company}: Technical indicators calculated")

print("\nFeature engineering completed")

In [ ]:
# Display sample data with new features
print("Sample data with technical indicators (Apple, last 5 rows):")
print(stock_data["Apple"].tail())
print(f"\nColumns: {list(stock_data['Apple'].columns)}")

## 5. Descriptive Analysis and Insights

In [ ]:
# Calculate key metrics for each company
def calculate_metrics(data, company_name):
    """
    Calculate key financial metrics for analysis
    """
    metrics = {
        'Company': company_name,
        'Start_Price': data['Close'].iloc[0],
        'End_Price': data['Close'].iloc[-1],
        'Min_Price': data['Close'].min(),
        'Max_Price': data['Close'].max(),
        'Avg_Price': data['Close'].mean(),
        'Price_Change': data['Close'].iloc[-1] - data['Close'].iloc[0],
        'Price_Change_Pct': ((data['Close'].iloc[-1] / data['Close'].iloc[0]) - 1) * 100,
        'Volatility': data['Close'].std(),
        'Volatility_Pct': (data['Close'].std() / data['Close'].mean()) * 100,
        'Avg_Daily_Return': data['Daily_Return'].mean(),
        'Avg_Volume': data['Volume'].mean(),
        'Max_Volume': data['Volume'].max(),
        'Min_Volume': data['Volume'].min(),
        'Avg_Price_Range': data['Price_Range'].mean(),
        'Positive_Days': (data['Daily_Return'] > 0).sum(),
        'Negative_Days': (data['Daily_Return'] < 0).sum(),
        'Total_Days': len(data)
    }
    return metrics

# Calculate metrics for all companies
metrics_list = []
for company, data in stock_data.items():
    metrics = calculate_metrics(data, company)
    metrics_list.append(metrics)

metrics_df = pd.DataFrame(metrics_list)

print("Key Financial Metrics Summary:")
print("=" * 100)
print(metrics_df.to_string())

In [ ]:
# Performance ranking
print("\n" + "="*50)
print("PERFORMANCE RANKINGS")
print("="*50)

print("\n1. Best Performers (by % change):")
top_performers = metrics_df.nlargest(3, 'Price_Change_Pct')[['Company', 'Price_Change_Pct']]
for idx, row in top_performers.iterrows():
    print(f"   {row['Company']}: +{row['Price_Change_Pct']:.2f}%")

print("\n2. Most Volatile (by volatility %):")
most_volatile = metrics_df.nlargest(3, 'Volatility_Pct')[['Company', 'Volatility_Pct']]
for idx, row in most_volatile.iterrows():
    print(f"   {row['Company']}: {row['Volatility_Pct']:.2f}%")

print("\n3. Highest Average Daily Return:")
highest_return = metrics_df.nlargest(3, 'Avg_Daily_Return')[['Company', 'Avg_Daily_Return']]
for idx, row in highest_return.iterrows():
    print(f"   {row['Company']}: {row['Avg_Daily_Return']:.4f}%")

print("\n4. Highest Trading Volume:")
highest_volume = metrics_df.nlargest(3, 'Avg_Volume')[['Company', 'Avg_Volume']]
for idx, row in highest_volume.iterrows():
    print(f"   {row['Company']}: {row['Avg_Volume']:,.0f}")

## 6. Comparative Analysis

In [ ]:
# Normalize prices to compare performance (base = 100 at start date)
normalized_data = {}

for company, data in stock_data.items():
    first_price = data['Close'].iloc[0]
    normalized_data[company] = (data['Close'] / first_price * 100)

# Create comparison DataFrame
comparison_df = pd.DataFrame(normalized_data)

print("Normalized Price Comparison (Base = 100 at start date)")
print("First 5 rows:")
print(comparison_df.head())
print("\nLast 5 rows:")
print(comparison_df.tail())
print(f"\nCurrent Performance (as of {comparison_df.index[-1].date()}):")
print(comparison_df.iloc[-1].sort_values(ascending=False))

In [ ]:
# Correlation analysis
print("\nPrice Correlation Matrix:")
print("="*50)

price_data = {company: data['Close'] for company, data in stock_data.items()}
price_df = pd.DataFrame(price_data)
correlation_matrix = price_df.corr()

print(correlation_matrix.round(3))

print("\nInterpretation:")
print("- Values close to 1.0: Strong positive correlation (move together)")
print("- Values close to 0.0: No correlation (independent movements)")
print("- Values close to -1.0: Strong negative correlation (move opposite)")

## 7. Visualization

In [ ]:
# Set up visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Stock Market Analysis: Tech Companies (6-Month Period)', fontsize=16, fontweight='bold')

# Plot 1: Price Trends
ax1 = axes[0, 0]
for company, data in stock_data.items():
    ax1.plot(data.index, data['Close'], label=company, linewidth=2)
ax1.set_title('Stock Price Trends', fontsize=12, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Price (USD)')
ax1.legend(loc='best', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Normalized Performance
ax2 = axes[0, 1]
for company in normalized_data:
    ax2.plot(normalized_data[company].index, normalized_data[company], label=company, linewidth=2)
ax2.set_title('Normalized Performance (Base = 100)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Normalized Price')
ax2.axhline(y=100, color='r', linestyle='--', alpha=0.5, label='Start Price')
ax2.legend(loc='best', fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: Price Change Percentage
ax3 = axes[1, 0]
companies_list = metrics_df['Company'].tolist()
price_changes = metrics_df['Price_Change_Pct'].tolist()
colors = ['green' if x > 0 else 'red' for x in price_changes]
ax3.bar(companies_list, price_changes, color=colors, alpha=0.7)
ax3.set_title('6-Month Price Change (%)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Change (%)')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Volatility Comparison
ax4 = axes[1, 1]
volatility = metrics_df['Volatility_Pct'].tolist()
ax4.bar(companies_list, volatility, color='steelblue', alpha=0.7)
ax4.set_title('Price Volatility (%)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Volatility (%)')
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('stock_analysis_overview.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'stock_analysis_overview.png'")

In [ ]:
# Detailed analysis for a specific company (Apple)
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Detailed Analysis: Apple Inc. (AAPL)', fontsize=14, fontweight='bold')

apple_data = stock_data['Apple']

# Price with Moving Averages
ax1 = axes[0]
ax1.plot(apple_data.index, apple_data['Close'], label='Close Price', linewidth=2, color='blue')
ax1.plot(apple_data.index, apple_data['MA20'], label='20-day MA', linewidth=1.5, color='orange', linestyle='--')
ax1.plot(apple_data.index, apple_data['MA50'], label='50-day MA', linewidth=1.5, color='red', linestyle='--')
ax1.fill_between(apple_data.index, apple_data['Close'], alpha=0.1, color='blue')
ax1.set_title('Price Trend with Moving Averages', fontsize=12, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Trading Volume
ax2 = axes[1]
ax2.bar(apple_data.index, apple_data['Volume'], color='steelblue', alpha=0.6, label='Daily Volume')
ax2.plot(apple_data.index, apple_data['Volume_MA20'], color='red', linewidth=2, label='20-day Volume MA')
ax2.set_title('Trading Volume', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Volume')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('apple_detailed_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Detailed Apple analysis saved as 'apple_detailed_analysis.png'")

## 8. Main Insights and Findings

In [ ]:
print("\n" + "="*70)
print("MAIN INSIGHTS AND FINDINGS")
print("="*70)

# Insight 1: Performance Leaders
print("\n1. PERFORMANCE LEADERS")
print("-" * 70)
best_performer = metrics_df.loc[metrics_df['Price_Change_Pct'].idxmax()]
worst_performer = metrics_df.loc[metrics_df['Price_Change_Pct'].idxmin()]
print(f"Best Performer: {best_performer['Company']} (+{best_performer['Price_Change_Pct']:.2f}%)")
print(f"Worst Performer: {worst_performer['Company']} ({worst_performer['Price_Change_Pct']:.2f}%)")
print(f"Performance Range: {best_performer['Price_Change_Pct'] - worst_performer['Price_Change_Pct']:.2f} percentage points")

# Insight 2: Volatility Analysis
print("\n2. VOLATILITY ANALYSIS")
print("-" * 70)
most_volatile = metrics_df.loc[metrics_df['Volatility_Pct'].idxmax()]
least_volatile = metrics_df.loc[metrics_df['Volatility_Pct'].idxmin()]
print(f"Most Volatile: {most_volatile['Company']} ({most_volatile['Volatility_Pct']:.2f}%)")
print(f"Least Volatile: {least_volatile['Company']} ({least_volatile['Volatility_Pct']:.2f}%)")
print(f"Volatility Range: {most_volatile['Volatility_Pct'] - least_volatile['Volatility_Pct']:.2f} percentage points")

# Insight 3: Trading Activity
print("\n3. TRADING ACTIVITY")
print("-" * 70)
highest_volume = metrics_df.loc[metrics_df['Avg_Volume'].idxmax()]
print(f"Highest Average Volume: {highest_volume['Company']} ({highest_volume['Avg_Volume']:,.0f} shares/day)")
print(f"Average Volume across all stocks: {metrics_df['Avg_Volume'].mean():,.0f} shares/day")

# Insight 4: Risk-Return Profile
print("\n4. RISK-RETURN PROFILE")
print("-" * 70)
metrics_df['Risk_Adjusted_Return'] = metrics_df['Price_Change_Pct'] / metrics_df['Volatility_Pct']
best_risk_adjusted = metrics_df.loc[metrics_df['Risk_Adjusted_Return'].idxmax()]
print(f"Best Risk-Adjusted Return: {best_risk_adjusted['Company']}")
print(f"  Return: {best_risk_adjusted['Price_Change_Pct']:.2f}%")
print(f"  Volatility: {best_risk_adjusted['Volatility_Pct']:.2f}%")
print(f"  Risk-Adjusted Ratio: {best_risk_adjusted['Risk_Adjusted_Return']:.4f}")

# Insight 5: Correlation
print("\n5. MARKET CORRELATION")
print("-" * 70)
avg_correlation = correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].mean()
print(f"Average Correlation between stocks: {avg_correlation:.3f}")
if avg_correlation > 0.7:
    print("Interpretation: Stocks move together (high systematic risk)")
elif avg_correlation > 0.4:
    print("Interpretation: Moderate positive correlation (some diversification benefit)")
else:
    print("Interpretation: Low correlation (good diversification potential)")

## 9. Limitations and Data Quality Assessment

In [ ]:
print("\n" + "="*70)
print("LIMITATIONS AND DATA QUALITY ASSESSMENT")
print("="*70)

print("\n1. DATA LIMITATIONS")
print("-" * 70)
print("• Time Period: 6-month analysis may not capture long-term trends")
print("• Market Conditions: Analysis period may include unusual market events")
print("• No Fundamental Data: Analysis focuses on price action, not company fundamentals")
print("• Survivorship Bias: Only includes companies still trading (no delisted stocks)")
print("• Sector Concentration: All companies are in technology sector")

print("\n2. METHODOLOGICAL LIMITATIONS")
print("-" * 70)
print("• Simple Moving Averages: No weighting for recent data")
print("• No Advanced Models: Does not include ARIMA, Prophet, or ML models")
print("• No Risk Metrics: Does not calculate Sharpe ratio, beta, or VaR")
print("• Correlation Assumptions: Assumes linear relationships")
print("• No Transaction Costs: Ignores fees and slippage in trading")

print("\n3. DATA QUALITY CHECKS")
print("-" * 70)
for company, data in stock_data.items():
    data_quality_score = 100
    issues = []
    
    # Check for gaps
    if len(data) < 150:  # Expected ~180 trading days
        data_quality_score -= 10
        issues.append(f"Fewer trading days than expected ({len(data)})")
    
    # Check for outliers
    daily_returns = data['Daily_Return'].dropna()
    outliers = ((daily_returns > daily_returns.mean() + 3*daily_returns.std()) | 
                (daily_returns < daily_returns.mean() - 3*daily_returns.std())).sum()
    if outliers > 5:
        data_quality_score -= 5
        issues.append(f"Potential outliers detected ({outliers} days)")
    
    print(f"\n{company}:")
    print(f"  Quality Score: {data_quality_score}/100")
    print(f"  Trading Days: {len(data)}")
    if issues:
        print(f"  Issues: {', '.join(issues)}")
    else:
        print(f"  Status: ✓ No quality issues detected")

## 10. Possible Improvements

In [ ]:
print("\n" + "="*70)
print("POSSIBLE IMPROVEMENTS AND FUTURE ENHANCEMENTS")
print("="*70)

improvements = [
    {
        'category': 'Technical Indicators',
        'items': [
            'Add RSI (Relative Strength Index) for overbought/oversold signals',
            'Implement MACD (Moving Average Convergence Divergence)',
            'Calculate Bollinger Bands for volatility assessment',
            'Add Stochastic Oscillator for momentum analysis'
        ]
    },
    {
        'category': 'Fundamental Analysis',
        'items': [
            'Integrate P/E ratios and earnings data',
            'Add dividend yield information',
            'Include debt-to-equity ratios',
            'Calculate ROE and ROA metrics'
        ]
    },
    {
        'category': 'Risk Analysis',
        'items': [
            'Calculate Sharpe ratio for risk-adjusted returns',
            'Compute beta relative to market index',
            'Estimate Value at Risk (VaR)',
            'Analyze maximum drawdown'
        ]
    },
    {
        'category': 'Predictive Modeling',
        'items': [
            'Implement ARIMA for time series forecasting',
            'Use Prophet for seasonal trend analysis',
            'Develop machine learning models (Random Forest, LSTM)',
            'Create ensemble models for better predictions'
        ]
    },
    {
        'category': 'Data Expansion',
        'items': [
            'Add more companies across different sectors',
            'Include cryptocurrency and commodity prices',
            'Integrate economic indicators (GDP, inflation, interest rates)',
            'Add news sentiment analysis'
        ]
    }
]

for improvement in improvements:
    print(f"\n{improvement['category']}:")
    for item in improvement['items']:
        print(f"  • {item}")

## 11. Conclusion

In [ ]:
print("\n" + "="*70)
print("ANALYSIS CONCLUSION")
print("="*70)

print("""
This analysis demonstrates a comprehensive Python-based workflow for stock market analysis:

1. DATA ACQUISITION: Successfully fetched 6 months of historical data for 8 tech companies
   from Yahoo Finance, ensuring data reliability and completeness.

2. DATA PROCESSING: Implemented robust data cleaning and validation procedures,
   removing inconsistencies and ensuring logical data integrity.

3. FEATURE ENGINEERING: Calculated technical indicators (moving averages, returns,
   volatility) to support analytical insights.

4. ANALYSIS: Performed descriptive and comparative analysis, identifying performance
   leaders, volatility patterns, and correlation structures.

5. VISUALIZATION: Created multiple visualizations to communicate findings effectively
   to the target audience.

6. PRODUCT DELIVERY: Developed an interactive Streamlit application enabling users
   to explore data dynamically and make informed investment decisions.

The tool successfully addresses the analytical problem by providing investors and
analysts with a user-friendly platform for comparative stock analysis, supporting
better-informed investment decisions through data-driven insights.
""")

print("\nAnalysis completed successfully!")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")